In [17]:
import numpy as np
import cupy as cp
from cupyx.scipy import ndimage
from tanalysis import improcess
import tifffile as tiff
from skimage.measure import label, regionprops, regionprops_table
from skimage import exposure, morphology
import os
import itertools

In [25]:
def LoG(sigma_x, sigma_y, sigma_z, n=7):
    z,y,x = cp.ogrid[-n//2:n//2+1, -n//2:n//2+1, -n//2:n//2+1]
    z_filter = cp.exp(-(z*z/(2*sigma_z**2)))
    y_filter = cp.exp(-(y*y/(2*sigma_y**2)))
    x_filter = cp.exp(-(x*x/(2*sigma_x**2)))
    final_filter = (1/(sigma_x*sigma_y*sigma_z*(2*np.pi)**(3/2)))*((1-x*x/(sigma_x**2))/(sigma_x**2)+(1-y*y/(sigma_y**2))/(sigma_y**2)+(1-z*z/(sigma_z**2))/(sigma_z**2))*(z_filter*x_filter*y_filter)
    return final_filter

In [26]:
def LoG_convolve(img, sigmas_x, sigma_y, sigma_z, n=7):
    filter_log = LoG(sigmas_x, sigma_y, sigma_z, n=7)
    image = ndimage.convolve(img, filter_log)
    image = cp.square(image)
    return cp.asarray(image)

In [ ]:
def fit_gaussian(sigma_x, sigma_y, sigma_z, n=7):
    z,y,x = cp.ogrid[-n//2:n//2+1, -n//2:n//2+1, -n//2:n//2+1]
    z_filter = cp.exp(-(z*z/(2*sigma_z**2)))
    y_filter = cp.exp(-(y*y/(2*sigma_y**2)))
    x_filter = cp.exp(-(x*x/(2*sigma_x**2)))
    final_filter = (x_filter*y_filter*z_filter)/(sigma_x*sigma_y*sigma_z*(2*np.pi)**(3/2))
    return final_filter

In [5]:
def label_cells(image:np.ndarray, sigmas:list, th:float):
    '''
    '''
    Z,Y,X = image.shape
    og_img = np.double(image)
    img = cp.array(og_img)

    gaussian = fit_gaussian(*sigmas)
    img = ndimage.convolve(img, gaussian)
    th1 = cp.max(img)*th #####################
    img_th1 = img>th1

    img = LoG_convolve(img, *sigmas)
    th2 = cp.max(img)*th #####################
    img_th2 = img>th2

    img_th1 = cp.asnumpy(img_th1)
    img_th2 = cp.asnumpy(img_th2)

    img = img_th1 ^ img_th2
    img = morphology.dilation(img)
    img = morphology.erosion(img)
    img = morphology.remove_small_holes(img)
    img_r1 = morphology.remove_small_objects(img)

    img = cp.asarray(np.float16(img_r1))
    img = LoG_convolve(img, *sigmas)
    img = cp.asnumpy(img)
    th3 = np.max(img)*th
    img_th3 = img>th3
    img_r2 = img_r1 ^ img_th3
    img = morphology.erosion(img_r2)
    img = morphology.remove_small_objects(img)
    img = label(img)
    return img

In [39]:
dirname = r"D:\pcanaleta\Thunder\260327\Compatibility\WT\Originals"
for file in os.listdir(dirname):
    path = os.path.join(dirname, file)
    if os.path.isdir(path):
        continue
    imgs, names, info = improcess.imread(path, False, False)
    T,Z,Y,X = imgs[0].shape
    result = np.empty(shape=(T,Z,Y,X), dtype=np.int16)
    i=0
    for img in imgs[0]:
        max_value = np.max(img)
        lab_img = img>max_value*0.05
        lab_img = morphology.remove_small_holes(lab_img)
        lab_img = morphology.remove_small_objects(lab_img)
        lab_img = morphology.dilation(lab_img)
        lab_img = morphology.erosion(lab_img)
        log_lab_img = np.double(lab_img)
        log_lab_img = cp.array(log_lab_img)
        log_lab_img = LoG_convolve(log_lab_img, 0.6, 0.6, 0.6, n=7)
        log_lab_img = cp.asnumpy(log_lab_img)
        lab_img = np.uint16(lab_img) & np.uint16(log_lab_img>np.mean(log_lab_img))
        result[i] = label(lab_img)
        i +=1
    tiff.imwrite(fr"D:\pcanaleta\Thunder\260327\Compatibility\WT\Results\{names[0]}.tiff", 
            np.uint16(result), 
            imagej=True,
            metadata={
                'axes':"TZYX"
            })